# VNI Data Cleaning and Forecasting

This notebook is a VNI-focused version of `clean_data.ipynb`. It loads `VNI.csv`, cleans numeric columns, and builds ARIMA/Prophet forecasts on the Close series.

In [1]:
import pandas as pd
from datetime import datetime
import time
import vnstock
from vnstock import Vnstock

📦 **Vnstock 4.0.4 is available**
Current: 4.0.2 (Python 3.11 (venv))
Update: `c:\Users\Tuan Minh Hoang\Time-Series-Forecasting-Apple-Stock-Price-Using-SARIMA-Prophet\.venv\Scripts\python.exe -m pip install vnstock --upgrade`
Release: https://vnstocks.com/docs/tai-lieu/lich-su-phien-ban

In [2]:
API = 'vnstock_68210bd47b80df1b69d8eebed54d6393'
from vnstock import register_user
register_user(api_key=API)

✓ API key đã được lưu thành công! (API key saved successfully!)
Bạn đang sử dụng Phiên bản cộng đồng (60 requests/phút)
(You are using Community version - 60 requests/minute)

Để tham gia gói thành viên tài trợ (To join sponsor membership):
  Truy cập: https://vnstocks.com/insiders-program
✓ API key đã được lưu thành công! (✓ API key saved successfully! vnst***6393
✓ Bạn đang sử dụng gói Cộng đồng (✓ You are using Community Edition - 60 requests/min)


True

# DATA VN-INDEX

In [3]:
def fetch_vnindex_nested(start_date_str):
    mkt = vnstock.Market()
    overall_start = pd.to_datetime(start_date_str)
    final_end = datetime.now()
    
    all_data = [] # Chứa toàn bộ dữ liệu từ 2012 đến nay
    
    print(f"Bắt đầu crawl dữ liệu từ {start_date_str} đến nay...")
    
    current_macro_start = overall_start
    
    # --- VÒNG LẶP NGOÀI (MACRO): CHIA CHU KỲ 2 NĂM ---
    while current_macro_start <= final_end:
        # Cộng 2 năm, trừ 1 ngày
        macro_end = current_macro_start + pd.DateOffset(years=2) - pd.Timedelta(days=1)
        if macro_end > final_end:
            macro_end = final_end
            
        print("\n" + "="*60)
        print(f"BẮT ĐẦU CHU KỲ: {current_macro_start.strftime('%Y-%m')} ĐẾN {macro_end.strftime('%Y-%m')}")
        print("="*60)
        
        current_micro_start = current_macro_start
        macro_chunks = [] # Chứa dữ liệu của riêng chu kỳ 2 năm hiện tại
        
        # --- VÒNG LẶP TRONG (MICRO): CHIA NHỎ 150 NGÀY ---
        while current_micro_start <= macro_end:
            micro_end = current_micro_start + pd.Timedelta(days=130)
            if micro_end > macro_end:
                micro_end = macro_end
                
            str_start = current_micro_start.strftime("%Y-%m-%d")
            str_end = micro_end.strftime("%Y-%m-%d")
            
            print(f"[{datetime.now().strftime('%H:%M:%S')}] {str_start} -> {str_end}", end=" ")
            
            try:
                # Gọi API
                df_chunk = mkt.index("VNINDEX").ohlcv(start=str_start, end=str_end)
                
                if df_chunk is not None and not df_chunk.empty:
                    macro_chunks.append(df_chunk)
                    print(f"-> OK ({len(df_chunk)} dòng)")
                else:
                    print("-> Trống")
                    
            except Exception as e:
                print(f"-> LỖI: {e}")
                
            # Tịnh tiến cho vòng lặp nhỏ tiếp theo
            current_micro_start = micro_end + pd.Timedelta(days=1)
            
            # Nghỉ ngơi 2s để tôn trọng giới hạn 20 request/phút
            if current_micro_start <= final_end:
                time.sleep(1)
                
        # --- KẾT THÚC VÒNG LẶP TRONG CỦA 1 CHU KỲ ---
        if macro_chunks:
            macro_df = pd.concat(macro_chunks)
            all_data.append(macro_df)
            print(f">>> Hoàn thành chu kỳ. Thu được tổng: {len(macro_df)} dòng.")
        
        if macro_end <= final_end:
            time.sleep(10)
        # Tịnh tiến sang chu kỳ 2 năm tiếp theo
        current_macro_start = macro_end + pd.Timedelta(days=1)
        
    # --- GỘP TOÀN BỘ DỮ LIỆU CUỐI CÙNG ---
    if all_data:
        print("\n" + "*"*60)
        print("ĐANG GỘP VÀ LÀM SẠCH TOÀN BỘ DỮ LIỆU LỊCH SỬ...")
        final_df = pd.concat(all_data)
        
        # Lọc trùng lặp tại các điểm nối và sắp xếp thời gian
        final_df = final_df.drop_duplicates(subset=['time'], keep='first')
        final_df = final_df.sort_values(by='time')
        final_df = final_df.reset_index(drop=True)
        
        print(f"HOÀN THÀNH XUẤT SẮC! Tổng số dòng VNI: {len(final_df)}")
        return final_df
    else:
        print("Không tải được dữ liệu.")
        return pd.DataFrame()

# --- Chạy thực tế ---
# Kéo từ 2013
vnindex_history = fetch_vnindex_nested("2013-01-01")

if not vnindex_history.empty:
    vnindex_history.to_csv("data/VNINDEX_Full_History.csv", index=False)
    print("Đã xuất file: data/VNINDEX_Full_History.csv")

Bắt đầu crawl dữ liệu từ 2013-01-01 đến nay...

BẮT ĐẦU CHU KỲ: 2013-01 ĐẾN 2014-12
[00:29:17] 2013-01-01 -> 2013-05-11 

-> OK (84 dòng)
[00:29:36] 2013-05-12 -> 2013-09-19 -> OK (93 dòng)
[00:29:37] 2013-09-20 -> 2014-01-28 -> OK (91 dòng)
[00:29:39] 2014-01-29 -> 2014-06-08 -> OK (83 dòng)
[00:29:41] 2014-06-09 -> 2014-10-17 -> OK (93 dòng)
[00:29:43] 2014-10-18 -> 2014-12-31 -> OK (53 dòng)
>>> Hoàn thành chu kỳ. Thu được tổng: 497 dòng.

BẮT ĐẦU CHU KỲ: 2015-01 ĐẾN 2016-12
[00:29:55] 2015-01-01 -> 2015-05-11 -> OK (81 dòng)
[00:29:56] 2015-05-12 -> 2015-09-19 -> OK (93 dòng)
[00:29:58] 2015-09-20 -> 2016-01-28 -> OK (93 dòng)
[00:29:59] 2016-01-29 -> 2016-06-07 -> OK (85 dòng)
[00:30:01] 2016-06-08 -> 2016-10-16 -> OK (92 dòng)
[00:30:02] 2016-10-17 -> 2016-12-31 -> OK (55 dòng)
>>> Hoàn thành chu kỳ. Thu được tổng: 499 dòng.

BẮT ĐẦU CHU KỲ: 2017-01 ĐẾN 2018-12
[00:30:14] 2017-01-01 -> 2017-05-11 -> OK (85 dòng)
[00:30:16] 2017-05-12 -> 2017-09-19 -> OK (92 dòng)
[00:30:17] 2017-09-20 -> 2018-01-28 -> OK (91 dòng)
[00:30:19] 2018-01-29 -> 2018-06-08 -> OK (87 dòng)
[00:30:21] 2018-06-09 -> 2018-10-1

# DATA S&P 500

In [11]:
def fetch_sp500_nested(start_date_str):
    overall_start = pd.to_datetime(start_date_str)
    final_end = datetime.now()
    
    all_data = [] # Chứa toàn bộ dữ liệu từ 2007 đến nay
    
    print(f"Bắt đầu crawl dữ liệu từ {start_date_str} đến nay...")
    
    current_macro_start = overall_start
    
    # --- VÒNG LẶP NGOÀI (MACRO): CHIA CHU KỲ 2 NĂM ---
    while current_macro_start <= final_end:
        # Cộng 2 năm, trừ 1 ngày
        macro_end = current_macro_start + pd.DateOffset(years=2) - pd.Timedelta(days=1)
        if macro_end > final_end:
            macro_end = final_end
            
        print("\n" + "="*60)
        print(f"BẮT ĐẦU CHU KỲ: {current_macro_start.strftime('%Y-%m')} ĐẾN {macro_end.strftime('%Y-%m')}")
        print("="*60)
        
        current_micro_start = current_macro_start
        macro_chunks = [] # Chứa dữ liệu của riêng chu kỳ 2 năm hiện tại
        
        # --- VÒNG LẶP TRONG (MICRO): CHIA NHỎ 130 NGÀY ---
        while current_micro_start <= macro_end:
            micro_end = current_micro_start + pd.Timedelta(days=130)
            if micro_end > macro_end:
                micro_end = macro_end
                
            str_start = current_micro_start.strftime("%Y-%m-%d")
            str_end = micro_end.strftime("%Y-%m-%d")
            
            print(f"[{datetime.now().strftime('%H:%M:%S')}] {str_start} -> {str_end}", end=" ")
            
            try:
                # Gọi API
                df_sp500 = Vnstock().world_index(symbol='INX', source='MSN').quote.history(start=str_start, end=str_end)
                
                if df_sp500 is not None and not df_sp500.empty:
                    macro_chunks.append(df_sp500)
                    print(f"-> OK ({len(df_sp500)} dòng)")
                else:
                    print("-> Trống")
                    
            except Exception as e:
                print(f"-> LỖI: {e}")
                
            # Tịnh tiến cho vòng lặp nhỏ tiếp theo
            current_micro_start = micro_end + pd.Timedelta(days=1)
            
            # Nghỉ ngơi 1s để tôn trọng giới hạn 20 request/phút
            if current_micro_start <= final_end:
                time.sleep(1)
                
        # --- KẾT THÚC VÒNG LẶP TRONG CỦA 1 CHU KỲ ---
        if macro_chunks:
            macro_df = pd.concat(macro_chunks)
            all_data.append(macro_df)
            print(f">>> Hoàn thành chu kỳ. Thu được tổng: {len(macro_df)} dòng.")
        
        if macro_end <= final_end:
            time.sleep(10)
        # Tịnh tiến sang chu kỳ 2 năm tiếp theo
        current_macro_start = macro_end + pd.Timedelta(days=1)
        
    # --- GỘP TOÀN BỘ DỮ LIỆU CUỐI CÙNG ---
    if all_data:
        print("\n" + "*"*60)
        print("ĐANG GỘP VÀ LÀM SẠCH TOÀN BỘ DỮ LIỆU LỊCH SỬ...")
        final_df = pd.concat(all_data)
        
        # Lọc trùng lặp tại các điểm nối và sắp xếp thời gian
        final_df = final_df.drop_duplicates(subset=['time'], keep='first')
        final_df = final_df.sort_values(by='time')
        final_df = final_df.reset_index(drop=True)
        
        print(f"HOÀN THÀNH XUẤT SẮC! Tổng số dòng SP500: {len(final_df)}")
        return final_df
    else:
        print("Không tải được dữ liệu.")
        return pd.DataFrame()

# --- Chạy thực tế ---
# Kéo từ 2016
sp500_history = fetch_sp500_nested('2016-04-01')

if not sp500_history.empty:
    sp500_history.to_csv("data/SP500_Full_History.csv", index=False)
    print("Đã xuất file: data/SP500_Full_History.csv")

2026-06-13 00:42:55 - vnstock.common.client - DEBUG - Mapped INX -> a33k6h
2026-06-13 00:42:55 - vnstock.common.data - DEBUG - No mapping for A33K6H, using as-is


Bắt đầu crawl dữ liệu từ 2016-04-01 đến nay...

BẮT ĐẦU CHU KỲ: 2016-04 ĐẾN 2018-03
[00:42:55] 2016-04-01 -> 2016-08-09 
  ╭──────────────────────────────────────────────────────────╮
  │  ⚠️  VNSTOCK DEPRECATION NOTICE (31/08/2025)             │
  │                                                          │
  │  Lớp Vnstock và các phương thức cũ (stock, fx, crypto,   │
  │  world_index, fund...) đã chính thức bị ngừng hỗ trợ.    │
  │                                                          │
  │  Để hệ thống ổn định và nhận được cập nhật mới nhất,     │
  │  vui lòng chuyển sang dùng bộ thư viện `vnstock.api`.    │
  │                                                          │
  │  👉 Xem hướng dẫn Migration: /vnstock-migration          │
  ╰──────────────────────────────────────────────────────────╯

Mẫu code chuyển đổi (Migration Example):
--------------------------------------
Cũ (Old):  stock = Vnstock().stock('ACB')
Mới (New): from vnstock.api.quote import Quote
          q = Quo

2026-06-13 00:42:57 - vnstock.common.client - DEBUG - Mapped INX -> a33k6h
2026-06-13 00:42:57 - vnstock.common.data - DEBUG - No mapping for A33K6H, using as-is


[00:42:57] 2016-08-10 -> 2016-12-18 
  ╭──────────────────────────────────────────────────────────╮
  │  ⚠️  VNSTOCK DEPRECATION NOTICE (31/08/2025)             │
  │                                                          │
  │  Lớp Vnstock và các phương thức cũ (stock, fx, crypto,   │
  │  world_index, fund...) đã chính thức bị ngừng hỗ trợ.    │
  │                                                          │
  │  Để hệ thống ổn định và nhận được cập nhật mới nhất,     │
  │  vui lòng chuyển sang dùng bộ thư viện `vnstock.api`.    │
  │                                                          │
  │  👉 Xem hướng dẫn Migration: /vnstock-migration          │
  ╰──────────────────────────────────────────────────────────╯

Mẫu code chuyển đổi (Migration Example):
--------------------------------------
Cũ (Old):  stock = Vnstock().stock('ACB')
Mới (New): from vnstock.api.quote import Quote
          q = Quote(symbol='ACB', source='VCI')

-> OK (91 dòng)


2026-06-13 00:42:59 - vnstock.common.client - DEBUG - Mapped INX -> a33k6h
2026-06-13 00:42:59 - vnstock.common.data - DEBUG - No mapping for A33K6H, using as-is


...

[00:44:32] 2025-09-07 -> 2026-01-15 
  ╭──────────────────────────────────────────────────────────╮
  │  ⚠️  VNSTOCK DEPRECATION NOTICE (31/08/2025)             │
  │                                                          │
  │  Lớp Vnstock và các phương thức cũ (stock, fx, crypto,   │
  │  world_index, fund...) đã chính thức bị ngừng hỗ trợ.    │
  │                                                          │
  │  Để hệ thống ổn định và nhận được cập nhật mới nhất,     │
  │  vui lòng chuyển sang dùng bộ thư viện `vnstock.api`.    │
  │                                                          │
  │  👉 Xem hướng dẫn Migration: /vnstock-migration          │
  ╰──────────────────────────────────────────────────────────╯

Mẫu code chuyển đổi (Migration Example):
--------------------------------------
Cũ (Old):  stock = Vnstock().stock('ACB')
Mới (New): from vnstock.api.quote import Quote
          q = Quote(symbol='ACB', source='VCI')

-> OK (91 dòng)


2026-06-13 00:44:34 - vnstock.common.client - DEBUG - Mapped INX -> a33k6h
2026-06-13 00:44:34 - vnstock.common.data - DEBUG - No mapping for A33K6H, using as-is


[00:44:34] 2026-01-16 -> 2026-03-31 
  ╭──────────────────────────────────────────────────────────╮
  │  ⚠️  VNSTOCK DEPRECATION NOTICE (31/08/2025)             │
  │                                                          │
  │  Lớp Vnstock và các phương thức cũ (stock, fx, crypto,   │
  │  world_index, fund...) đã chính thức bị ngừng hỗ trợ.    │
  │                                                          │
  │  Để hệ thống ổn định và nhận được cập nhật mới nhất,     │
  │  vui lòng chuyển sang dùng bộ thư viện `vnstock.api`.    │
  │                                                          │
  │  👉 Xem hướng dẫn Migration: /vnstock-migration          │
  ╰──────────────────────────────────────────────────────────╯

Mẫu code chuyển đổi (Migration Example):
--------------------------------------
Cũ (Old):  stock = Vnstock().stock('ACB')
Mới (New): from vnstock.api.quote import Quote
          q = Quote(symbol='ACB', source='VCI')

-> OK (51 dòng)
>>> Hoàn thành chu kỳ. Thu được tổng

2026-06-13 00:44:46 - vnstock.common.client - DEBUG - Mapped INX -> a33k6h
2026-06-13 00:44:46 - vnstock.common.data - DEBUG - No mapping for A33K6H, using as-is



BẮT ĐẦU CHU KỲ: 2026-04 ĐẾN 2026-06
[00:44:46] 2026-04-01 -> 2026-06-13 
  ╭──────────────────────────────────────────────────────────╮
  │  ⚠️  VNSTOCK DEPRECATION NOTICE (31/08/2025)             │
  │                                                          │
  │  Lớp Vnstock và các phương thức cũ (stock, fx, crypto,   │
  │  world_index, fund...) đã chính thức bị ngừng hỗ trợ.    │
  │                                                          │
  │  Để hệ thống ổn định và nhận được cập nhật mới nhất,     │
  │  vui lòng chuyển sang dùng bộ thư viện `vnstock.api`.    │
  │                                                          │
  │  👉 Xem hướng dẫn Migration: /vnstock-migration          │
  ╰──────────────────────────────────────────────────────────╯

Mẫu code chuyển đổi (Migration Example):
--------------------------------------
Cũ (Old):  stock = Vnstock().stock('ACB')
Mới (New): from vnstock.api.quote import Quote
          q = Quote(symbol='ACB', source='VCI')

-> OK (50 dòng)

In [13]:
sp500_history = pd.read_csv("data/sp500.csv", sep=";")
sp500_history = sp500_history.rename(
    columns={
        "Date": "time",        
        "Open": "open",
        "High": "high",
        "Low": "low",
        "Close": "close",        
        "Volume": "volume",        
    }
)

sp500_history["time"] = pd.to_datetime(sp500_history["time"], format="%d/%m/%Y") + pd.Timedelta(hours=7)
sp500_history["open"] = pd.to_numeric(sp500_history["open"].astype(str).str.replace(",", ""), errors="coerce")
sp500_history["high"] = pd.to_numeric(sp500_history["high"].astype(str).str.replace(",", ""), errors="coerce")
sp500_history["low"] = pd.to_numeric(sp500_history["low"].astype(str).str.replace(",", ""), errors="coerce")
sp500_history["close"] = pd.to_numeric(sp500_history["close"].astype(str).str.replace(",", ""), errors="coerce")
sp500_history["volume"] = sp500_history["volume"].map(_parse_volume)
sp500_history = sp500_history[["time", "open", "high", "low", "close", "volume"]].dropna(subset=["time", "open", "high", "low", "close"])
# Cắt sp500.csv chỉ lấy dữ liệu trước 2016-04-01 để tránh lặp với dữ liệu API từ 2016-04-01
sp500_history = sp500_history[sp500_history["time"] < "2016-05-13"]

nested_history = pd.read_csv("data/SP500_Full_History.csv")
nested_history["time"] = pd.to_datetime(nested_history["time"])
nested_history = nested_history[["time", "open", "high", "low", "close", "volume"]].dropna(subset=["time", "open", "high", "low", "close"])
# Chỉ lấy dữ liệu nested từ 2016-04-01 trở đi (API fetch từ ngày này)
nested_history = nested_history[nested_history["time"] >= "2016-05-13"]

merged_sp500_history = pd.concat([sp500_history, nested_history], ignore_index=True)
merged_sp500_history["time"] = pd.to_datetime(merged_sp500_history["time"])
merged_sp500_history = merged_sp500_history.dropna(subset=["time", "open", "high", "low", "close"])
merged_sp500_history = merged_sp500_history.drop_duplicates(subset=["time"], keep="last")
merged_sp500_history = merged_sp500_history.sort_values(by="time").reset_index(drop=True)

merged_sp500_history.to_csv("data/SP500_Full_History.csv", index=False)
print(f"Đã ghép sp500.csv vào dữ liệu nested. Tổng số dòng: {len(merged_sp500_history)}")
print(f"sp500.csv tới 2016-05-13: {len(sp500_history)} dòng")
print(f"nested từ 2016-05-13: {len(nested_history)} dòng")
print("Đã ghi đè file: data/SP500_Full_History.csv")

Đã ghép sp500.csv vào dữ liệu nested. Tổng số dòng: 3359
sp500.csv tới 2016-05-13: 847 dòng
nested từ 2016-05-13: 2512 dòng
Đã ghi đè file: data/SP500_Full_History.csv


# DATA USD-VND

In [14]:
def fetch_usdvnd_nested(start_date_str):
    overall_start = pd.to_datetime(start_date_str)
    final_end = datetime.now()
    
    all_data = [] # Chứa toàn bộ dữ liệu từ 2007 đến nay
    
    print(f"Bắt đầu crawl dữ liệu từ {start_date_str} đến nay...")
    
    current_macro_start = overall_start
    
    # --- VÒNG LẶP NGOÀI (MACRO): CHIA CHU KỲ 2 NĂM ---
    while current_macro_start <= final_end:
        # Cộng 2 năm, trừ 1 ngày
        macro_end = current_macro_start + pd.DateOffset(years=2) - pd.Timedelta(days=1)
        if macro_end > final_end:
            macro_end = final_end
            
        print("\n" + "="*60)
        print(f"BẮT ĐẦU CHU KỲ: {current_macro_start.strftime('%Y-%m')} ĐẾN {macro_end.strftime('%Y-%m')}")
        print("="*60)
        
        current_micro_start = current_macro_start
        macro_chunks = [] # Chứa dữ liệu của riêng chu kỳ 2 năm hiện tại
        
        # --- VÒNG LẶP TRONG (MICRO): CHIA NHỎ 130 NGÀY ---
        while current_micro_start <= macro_end:
            micro_end = current_micro_start + pd.Timedelta(days=130)
            if micro_end > macro_end:
                micro_end = macro_end
                
            str_start = current_micro_start.strftime("%Y-%m-%d")
            str_end = micro_end.strftime("%Y-%m-%d")
            
            print(f"[{datetime.now().strftime('%H:%M:%S')}] {str_start} -> {str_end}", end=" ")
            
            try:
                # Gọi API
                df_exchange = Vnstock().fx(symbol='USDVND', source='MSN').quote.history(start=str_start, end=str_end, interval='1D')
                
                if df_exchange is not None and not df_exchange.empty:
                    macro_chunks.append(df_exchange)
                    print(f"-> OK ({len(df_exchange)} dòng)")
                else:
                    print("-> Trống")
                    
            except Exception as e:
                print(f"-> LỖI: {e}")
                
            # Tịnh tiến cho vòng lặp nhỏ tiếp theo
            current_micro_start = micro_end + pd.Timedelta(days=1)
            
            # Nghỉ ngơi 1s để tôn trọng giới hạn 20 request/phút
            if current_micro_start <= final_end:
                time.sleep(1)
                
        # --- KẾT THÚC VÒNG LẶP TRONG CỦA 1 CHU KỲ ---
        if macro_chunks:
            macro_df = pd.concat(macro_chunks)
            all_data.append(macro_df)
            print(f">>> Hoàn thành chu kỳ. Thu được tổng: {len(macro_df)} dòng.")
        
        if macro_end <= final_end:
            time.sleep(10)
        # Tịnh tiến sang chu kỳ 2 năm tiếp theo
        current_macro_start = macro_end + pd.Timedelta(days=1)
        
    # --- GỘP TOÀN BỘ DỮ LIỆU CUỐI CÙNG ---
    if all_data:
        print("\n" + "*"*60)
        print("ĐANG GỘP VÀ LÀM SẠCH TOÀN BỘ DỮ LIỆU LỊCH SỬ...")
        final_df = pd.concat(all_data)
        
        # Lọc trùng lặp tại các điểm nối và sắp xếp thời gian
        final_df = final_df.drop_duplicates(subset=['time'], keep='first')
        final_df = final_df.sort_values(by='time')
        final_df = final_df.reset_index(drop=True)
        
        print(f"HOÀN THÀNH XUẤT SẮC! Tổng số dòng USDVND: {len(final_df)}")
        return final_df
    else:
        print("Không tải được dữ liệu.")
        return pd.DataFrame()

# --- Chạy thực tế ---
# Kéo từ 2013
usdvnd_history = fetch_usdvnd_nested('2013-01-01')

if not usdvnd_history.empty:
    usdvnd_history.to_csv("data/USDVND_Full_History.csv", index=False)
    print("Đã xuất file: data/USDVND_Full_History.csv")

2026-06-13 00:46:33 - vnstock.common.client - DEBUG - Mapped USDVND -> avyufr
2026-06-13 00:46:33 - vnstock.common.data - DEBUG - No mapping for AVYUFR, using as-is


Bắt đầu crawl dữ liệu từ 2013-01-01 đến nay...

BẮT ĐẦU CHU KỲ: 2013-01 ĐẾN 2014-12
[00:46:33] 2013-01-01 -> 2013-05-11 
  ╭──────────────────────────────────────────────────────────╮
  │  ⚠️  VNSTOCK DEPRECATION NOTICE (31/08/2025)             │
  │                                                          │
  │  Lớp Vnstock và các phương thức cũ (stock, fx, crypto,   │
  │  world_index, fund...) đã chính thức bị ngừng hỗ trợ.    │
  │                                                          │
  │  Để hệ thống ổn định và nhận được cập nhật mới nhất,     │
  │  vui lòng chuyển sang dùng bộ thư viện `vnstock.api`.    │
  │                                                          │
  │  👉 Xem hướng dẫn Migration: /vnstock-migration          │
  ╰──────────────────────────────────────────────────────────╯

Mẫu code chuyển đổi (Migration Example):
--------------------------------------
Cũ (Old):  stock = Vnstock().stock('ACB')
Mới (New): from vnstock.api.quote import Quote
          q = Quo

2026-06-13 00:46:39 - vnstock.common.client - DEBUG - Mapped USDVND -> avyufr
2026-06-13 00:46:39 - vnstock.common.data - DEBUG - No mapping for AVYUFR, using as-is


[00:46:39] 2013-05-12 -> 2013-09-19 
  ╭──────────────────────────────────────────────────────────╮
  │  ⚠️  VNSTOCK DEPRECATION NOTICE (31/08/2025)             │
  │                                                          │
  │  Lớp Vnstock và các phương thức cũ (stock, fx, crypto,   │
  │  world_index, fund...) đã chính thức bị ngừng hỗ trợ.    │
  │                                                          │
  │  Để hệ thống ổn định và nhận được cập nhật mới nhất,     │
  │  vui lòng chuyển sang dùng bộ thư viện `vnstock.api`.    │
  │                                                          │
  │  👉 Xem hướng dẫn Migration: /vnstock-migration          │
  ╰──────────────────────────────────────────────────────────╯

Mẫu code chuyển đổi (Migration Example):
--------------------------------------
Cũ (Old):  stock = Vnstock().stock('ACB')
Mới (New): from vnstock.api.quote import Quote
          q = Quote(symbol='ACB', source='VCI')

-> LỖI: RetryError[<Future at 0x22348957b10 state=fi

2026-06-13 00:46:46 - vnstock.common.client - DEBUG - Mapped USDVND -> avyufr
2026-06-13 00:46:46 - vnstock.common.data - DEBUG - No mapping for AVYUFR, using as-is


...

[00:49:32] 2025-09-20 -> 2026-01-28 
  ╭──────────────────────────────────────────────────────────╮
  │  ⚠️  VNSTOCK DEPRECATION NOTICE (31/08/2025)             │
  │                                                          │
  │  Lớp Vnstock và các phương thức cũ (stock, fx, crypto,   │
  │  world_index, fund...) đã chính thức bị ngừng hỗ trợ.    │
  │                                                          │
  │  Để hệ thống ổn định và nhận được cập nhật mới nhất,     │
  │  vui lòng chuyển sang dùng bộ thư viện `vnstock.api`.    │
  │                                                          │
  │  👉 Xem hướng dẫn Migration: /vnstock-migration          │
  ╰──────────────────────────────────────────────────────────╯

Mẫu code chuyển đổi (Migration Example):
--------------------------------------
Cũ (Old):  stock = Vnstock().stock('ACB')
Mới (New): from vnstock.api.quote import Quote
          q = Quote(symbol='ACB', source='VCI')

-> OK (92 dòng)


2026-06-13 00:49:34 - vnstock.common.client - DEBUG - Mapped USDVND -> avyufr
2026-06-13 00:49:34 - vnstock.common.data - DEBUG - No mapping for AVYUFR, using as-is


[00:49:34] 2026-01-29 -> 2026-06-08 
  ╭──────────────────────────────────────────────────────────╮
  │  ⚠️  VNSTOCK DEPRECATION NOTICE (31/08/2025)             │
  │                                                          │
  │  Lớp Vnstock và các phương thức cũ (stock, fx, crypto,   │
  │  world_index, fund...) đã chính thức bị ngừng hỗ trợ.    │
  │                                                          │
  │  Để hệ thống ổn định và nhận được cập nhật mới nhất,     │
  │  vui lòng chuyển sang dùng bộ thư viện `vnstock.api`.    │
  │                                                          │
  │  👉 Xem hướng dẫn Migration: /vnstock-migration          │
  ╰──────────────────────────────────────────────────────────╯

Mẫu code chuyển đổi (Migration Example):
--------------------------------------
Cũ (Old):  stock = Vnstock().stock('ACB')
Mới (New): from vnstock.api.quote import Quote
          q = Quote(symbol='ACB', source='VCI')

-> OK (85 dòng)


2026-06-13 00:49:36 - vnstock.common.client - DEBUG - Mapped USDVND -> avyufr
2026-06-13 00:49:36 - vnstock.common.data - DEBUG - No mapping for AVYUFR, using as-is


[00:49:36] 2026-06-09 -> 2026-06-13 
  ╭──────────────────────────────────────────────────────────╮
  │  ⚠️  VNSTOCK DEPRECATION NOTICE (31/08/2025)             │
  │                                                          │
  │  Lớp Vnstock và các phương thức cũ (stock, fx, crypto,   │
  │  world_index, fund...) đã chính thức bị ngừng hỗ trợ.    │
  │                                                          │
  │  Để hệ thống ổn định và nhận được cập nhật mới nhất,     │
  │  vui lòng chuyển sang dùng bộ thư viện `vnstock.api`.    │
  │                                                          │
  │  👉 Xem hướng dẫn Migration: /vnstock-migration          │
  ╰──────────────────────────────────────────────────────────╯

Mẫu code chuyển đổi (Migration Example):
--------------------------------------
Cũ (Old):  stock = Vnstock().stock('ACB')
Mới (New): from vnstock.api.quote import Quote
          q = Quote(symbol='ACB', source='VCI')

-> OK (3 dòng)
>>> Hoàn thành chu kỳ. Thu được tổng:

In [16]:
usdvnd_history = pd.read_csv("data/usdvnd.csv", sep=";")
usdvnd_history = usdvnd_history.rename(
    columns={
        "Date": "time",        
        "Open": "open",
        "High": "high",
        "Low": "low",
        "Close": "close",        
        "Volume": "volume",        
    }
)

usdvnd_history["time"] = pd.to_datetime(usdvnd_history["time"], format="%d/%m/%Y") + pd.Timedelta(hours=7)
usdvnd_history["open"] = pd.to_numeric(usdvnd_history["open"].astype(str).str.replace(",", ""), errors="coerce")
usdvnd_history["high"] = pd.to_numeric(usdvnd_history["high"].astype(str).str.replace(",", ""), errors="coerce")
usdvnd_history["low"] = pd.to_numeric(usdvnd_history["low"].astype(str).str.replace(",", ""), errors="coerce")
usdvnd_history["close"] = pd.to_numeric(usdvnd_history["close"].astype(str).str.replace(",", ""), errors="coerce")
usdvnd_history["volume"] = usdvnd_history["volume"].map(_parse_volume)
usdvnd_history = usdvnd_history[["time", "open", "high", "low", "close", "volume"]].dropna(subset=["time", "open", "high", "low", "close"])
# Cắt usdvnd.csv chỉ lấy dữ liệu trước 2016-05-13 để tránh lặp với dữ liệu API từ 2016-05-13
usdvnd_history = usdvnd_history[usdvnd_history["time"] < "2016-05-13"]

nested_history = pd.read_csv("data/USDVND_Full_History.csv")
nested_history["time"] = pd.to_datetime(nested_history["time"])
nested_history = nested_history[["time", "open", "high", "low", "close", "volume"]].dropna(subset=["time", "open", "high", "low", "close"])
# Chỉ lấy dữ liệu nested từ 2016-05-13 trở đi (API fetch từ ngày này)
nested_history = nested_history[nested_history["time"] >= "2016-05-13"]

merged_usdvnd_history = pd.concat([usdvnd_history, nested_history], ignore_index=True)
merged_usdvnd_history["time"] = pd.to_datetime(merged_usdvnd_history["time"])
merged_usdvnd_history = merged_usdvnd_history.dropna(subset=["time", "open", "high", "low", "close"])
merged_usdvnd_history = merged_usdvnd_history.drop_duplicates(subset=["time"], keep="last")
merged_usdvnd_history = merged_usdvnd_history.sort_values(by="time").reset_index(drop=True)

merged_usdvnd_history.to_csv("data/USDVND_Full_History.csv", index=False)
print(f"Đã ghép usdvnd.csv vào dữ liệu nested. Tổng số dòng: {len(merged_usdvnd_history)}")
print(f"usdvnd.csv tới 2016-05-13: {len(usdvnd_history)} dòng")
print(f"nested từ 2016-05-13: {len(nested_history)} dòng")
print("Đã ghi đè file: data/USDVND_Full_History.csv")

Đã ghép usdvnd.csv vào dữ liệu nested. Tổng số dòng: 3413
usdvnd.csv tới 2016-05-13: 878 dòng
nested từ 2016-05-13: 2535 dòng
Đã ghi đè file: data/USDVND_Full_History.csv
